#### MetaQuest3 *controller* (joystick) teleoperation -> `FFW SG2` (sim only)

Uses the Quest controllers (thumbstick/trigger/squeeze), not hand tracking.
Data comes in over WebXR via a local Vuer HTTPS server (same mechanism as
ROBOTIS' `vr_publisher_sg2.py`, but headless / no ROS2) and drives the same
collision-aware wrist IK used by `sim_MetaQuest3_sg2_teleop.ipynb`.

**Controls**
- Squeeze (grip) a controller to engage/attach that arm; it will smoothly
  clutch onto your current controller pose. Release to disengage.
- Trigger opens/closes that hand's SG2 gripper.
- Move/rotate the controller (while gripping) to drive the wrist target.


In [1]:
%run ../package/init_project.py

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *

app = get_qt_app()



[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation
Basic packages initialized.
 import numpy as np
 import matplotlib.pyplot as plt
 import scipy as sp
 from PIL import Image
[ipython] settings (retina, inline, qt) initialized.
[numpy] settings initialized.
[matplotlib] settings initialized.
[qt] package initialized.


In [2]:
native_backend = build_sg2_collision_aware_ik_backend(
    clean   = False,
    verbose = True,
    required = True,)
print("bind info:",native_backend.native_info())


[build_native]
  module      : _mujoco_collision_aware_ik_bind
  output      : mujoco_sim/so/_mujoco_collision_aware_ik_bind.cpython-310-darwin.so
  compiler    : clang++
  project cpp:
    [01] mujoco_sim/cpp/bindings.cpp
    [02] mujoco_sim/cpp/signed_distance_bindings.cpp
    [03] mujoco_sim/cpp/signed_distance.cpp
    [04] mujoco_sim/cpp/ik_bindings.cpp
    [05] mujoco_sim/cpp/ik.cpp
  project hpp:
    [01] mujoco_sim/cpp/signed_distance_bindings.hpp
    [02] mujoco_sim/cpp/signed_distance.hpp
    [03] mujoco_sim/cpp/ik_bindings.hpp
    [04] mujoco_sim/cpp/ik.hpp
    [05] mujoco_sim/cpp/mujoco_handles.hpp
  defines:
    [01] RI_NATIVE_ENABLE_SIGNED_DISTANCE
    [02] RI_NATIVE_ENABLE_IK
    [03] RI_NATIVE_MODULE_NAME=_mujoco_collision_aware_ik_bind
  nanobind cpp:
    [01] nanobind/src/nb_combined.cpp
  commands    : hidden; pass show_commands=True to print full commands
[build_native] compile
  [01/06] mujoco_sim/cpp/bindings.cpp -> build/native/bindings.o
  [02/06] mujoco_sim/cpp/

##### Start the Vuer controller server

If `cert.pem`/`key.pem` are missing next to this notebook, a self-signed
certificate is generated automatically (WebXR requires HTTPS). Open the
printed URL in the Meta Quest browser and accept the certificate warning,
then click "Enter VR".

In [ ]:
from pathlib import Path
import subprocess

cert_file = Path("cert.pem").resolve()
key_file = Path("key.pem").resolve()

if not (cert_file.exists() and key_file.exists()):
    print("Self-signed TLS cert not found - generating one for the Vuer HTTPS server...")
    subprocess.run([
        "openssl", "req", "-x509", "-newkey", "rsa:2048", "-nodes",
        "-keyout", str(key_file), "-out", str(cert_file),
        "-days", "365", "-subj", "/CN=localhost",
    ], check=True)

controller_bridge = create_sg2_vuer_controller_bridge(
    host      = "0.0.0.0",
    port      = 8012,
    cert_file = cert_file,
    key_file  = key_file,
    fps       = 30,)
print(f"Vuer controller server: {controller_bridge.url}")
print("Open this URL in the Meta Quest browser (accept the self-signed cert warning), then Enter VR.")
print("[MARKER] bridge cell v2 executed - this is a FRESH bridge instance, id=%d" % id(controller_bridge))


Vuer controller server: https://0.0.0.0:8012
Open this URL in the Meta Quest browser (accept the self-signed cert warning), then Enter VR.
[MARKER] bridge cell v2 executed - this is a FRESH bridge instance, id=4365761744
Vuer Server

Local:   https://vuer.ai?ws=wss://localhost:8012
Network: https://vuer.ai?ws=wss://192.168.6.100:8012

Workspace:

 · file:///Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/sim_notebook
-> https://vuer.ai/workspace



websocket is connected. id:96369cf9-c9cc-4811-8f97-efea8a49de52
default socket worker is up, adding clientEvents 
Uplink task running. id:96369cf9-c9cc-4811-8f97-efea8a49de52
WebSocket connection closed


##### Debug: is Quest controller data actually arriving?

Run this any time (independently of the main loop) to check the Vuer
connection and raw event payload - useful if grip/trigger don't seem to
do anything. If the main loop below is currently running, stop it first
(interrupt the kernel) before running this cell.

In [4]:
print_controller_bridge_debug(controller_bridge)


connected: True
body events received: 164 | last type: BODY_MOVE
controller events received: 164 | last type: CONTROLLER_MOVE
last controller event keys: ['left', 'leftState', 'right', 'rightState']
leftState keys: ['trigger', 'squeeze', 'touchpad', 'thumbstick', 'aButton', 'bButton', 'triggerValue', 'squeezeValue', 'touchpadValue', 'thumbstickValue']
rightState keys: ['trigger', 'squeeze', 'touchpad', 'thumbstick', 'aButton', 'bButton', 'triggerValue', 'squeezeValue', 'touchpadValue', 'thumbstickValue']
left_matrix is None: False | right_matrix is None: False
left_state: {'trigger': False, 'squeeze': False, 'touchpad': False, 'thumbstick': False, 'aButton': False, 'bButton': False, 'triggerValue': 0, 'squeezeValue': 0, 'touchpadValue': [0, 0], 'thumbstickValue': [0, 0]}
right_state: {'trigger': False, 'squeeze': False, 'touchpad': False, 'thumbstick': False, 'aButton': False, 'bButton': False, 'triggerValue': 0, 'squeezeValue': 0, 'touchpadValue': [0, 0], 'thumbstickValue': [0, 0]}


In [5]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)


[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].


In [6]:
input_hz = 20
render_hz = 20
max_missing_quest_ticks = quest_missing_tick_limit(
    input_hz              = input_hz,
    missing_grace_seconds = 1.0,)

hand_types = list(SG2_HAND_TYPES)
hand_labels = dict(SG2_HAND_LABELS)

# Reset - start from the robot's real bringup pose (qpos/sg2_joint_qpos.json:"bringup_start"),
# same starting pose the other sim notebooks (joint_control-based) use, instead of an
# arbitrary straight-down test pose.
import json as _json
_bringup_data = _json.loads((PROJECT_DIR / "qpos" / "sg2_joint_qpos.json").read_text())
_bringup_joint_names = list(_bringup_data["joint_names"])
_bringup_qpos = np.array(_bringup_data["qpos"],dtype=np.float64)  # canonical default pose (same key joint_control notebooks load by default)

close_objects("rp")
env.reset()
env.set_p("base_link","body",(0.0,0.0,0.01))
env.forward(q=_bringup_qpos,joint_names=_bringup_joint_names)
print("[MARKER] reset cell v3 (qpos-key bringup pose) executed. arm_l_joint1=%.3f arm_l_joint4=%.3f" % (env.get_qpos(joint_names=["arm_l_joint1"])[0], env.get_qpos(joint_names=["arm_l_joint4"])[0]))

command_joint_names = filter_existing_joint_names(env,SG2_REAL_COMMAND_JOINT_NAMES)
q_use = env.get_qpos(joint_names=command_joint_names)
q_use_min = env.get_joint_mins(command_joint_names)
q_use_max = env.get_joint_maxs(command_joint_names)

collision_geom_pairs,collision_candidate_info = create_sg2_collision_candidate_geom_pairs(
    env               = env,
    init_exclude_dist = 0.05,)
print_sg2_collision_candidate_summary(collision_candidate_info)

rp = MuJoCoRenderProcess(
    0.22,0.00,1.0,1.0,
    xml_path    = xml_path,
    title       = "MetaQuest3 controller IK -> FFW SG2",
    render_hz   = render_hz,
    max_sec     = 300.0,
    profile     = False,
    transparent = True,)
rp.start()

collision_ik_solvers,arm_joint_names = create_sg2_collision_aware_arm_ik_solvers(
    env                = env,
    mujoco_parser_cls  = MuJoCoParser,
    body_ik_solver_cls = BodyPositionIKSolverRevPriBase,
    enable_depenetration = False,)
active_ik_solver = collision_ik_solvers["tracking"]
collision_ik_result = make_idle_collision_ik_result(active_ik_solver,safe_margin=0.05)

# Anchor: controller poses are placed in MuJoCo world frame relative to this body,
# the same way the hand-tracking notebook anchors to "head_link2".
start_T_anchor = env.get_T(name="head_link2",type="body")

T_quest_wrist = {hand_type:None for hand_type in hand_types}
T_raw_wrist_target = {hand_type:None for hand_type in hand_types}
T_attach_wrist_target = {hand_type:None for hand_type in hand_types}
T_virtual_quest_wrist = {hand_type:None for hand_type in hand_types}

quest_missing_ticks = {hand_type:max_missing_quest_ticks+1 for hand_type in hand_types}
quest_all_missing_ticks = max_missing_quest_ticks+1
quest_attach_state = {
    hand_type:create_quest_wrist_attach_state(
        hand_type             = hand_type,
        attach_seconds        = SG2_CONTROLLER_ATTACH_SECONDS,
        attach_start_distance = SG2_CONTROLLER_ATTACH_START_DISTANCE,)
    for hand_type in hand_types
}
last_solved_wrist_target = {hand_type:None for hand_type in hand_types}
arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
wrist_target_motion = {hand_type:(np.inf,np.inf) for hand_type in hand_types}
gripper_q_by_hand = {hand_type:0.0 for hand_type in hand_types}
gripper_info_by_hand = {hand_type:{} for hand_type in hand_types}

# Per-hand orientation offset between "controller grip" and "SG2 wrist" frames.
# Only affects feel at the moment of attach (the offset is captured then, and
# rotation tracks 1:1 afterwards) - tune SG2_CONTROLLER_ROTATION_OFFSET_DEG in
# utils.py if left/right feel mirrored or a rotation axis feels inverted.
rotation_offset_R_by_hand = {
    hand_type: euler_xyz_deg_to_R(*SG2_CONTROLLER_ROTATION_OFFSET_DEG[hand_type])
    for hand_type in hand_types
}

for hand_type in hand_types:
    set_sg2_gripper_qpos(env,hand_type,gripper_q_by_hand[hand_type])

# Loop
tmr_update = SimpleTimer(name="Update",Hz=input_hz,verbose=True)
tmr_render = SimpleTimer(name="Render",Hz=render_hz,verbose=True)
tmr_update.start()
tmr_render.start()
env.reset_wall_time()

for hand_type in hand_types:
    print("%s arm IK joints:"%hand_type,len(arm_joint_names[hand_type]),arm_joint_names[hand_type])
print("collision IK joints:",active_ik_solver.joints_use)
print("Depenetration IK: disabled.")
print("Controller squeeze (grip) -> engage/attach that arm; release -> detach.")
print("Controller trigger -> SG2 gripper open/close.")


[MARKER] reset cell v3 (qpos-key bringup pose) executed. arm_l_joint1=1.570 arm_l_joint4=-2.800
Collision candidates
--------------------
robot collision geom:[34]
collision geom:[35]
candidate geom pairs:[436]
excluded initial geom pairs:[94]
render process pid:[7592] port:[59042] target_hz:[20.0]
render process log:[/var/folders/2s/nw51l0ms6fb_ptx8g18y4cm00000gn/T/ri_motion_render_1vx4i30w.log]
MuJoCo model loaded from [/Users/aiserving/Desktop/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
[Update] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Render] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Update] Start ([20.0]HZ. MAX_SEC:[inf]sec).
[Render] Start ([20.0]HZ. MAX_SEC:[inf]sec).
left arm IK joints: 7 ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l_joint5', 'arm_l_joint6', 'arm_l_joint7']
right arm IK joints: 7 ['arm_r_joint1', 'arm_r_joint2', 'arm_r_joint3', 'arm_r_joint4', 'arm_r_joint5', 'arm_r_joint6',

In [7]:
try:
    while rp.is_running():
        update_qt()
        env.increase_wall_time()

        if tmr_update.do_run():
            dt = getattr(tmr_update,"dt",1.0/input_hz)

            # Read MetaQuest3 controller pose. Squeeze (grip) gates engagement,
            # standing in for the hand-tracking pipeline's ZMQ "packet received".
            quest_updated = {hand_type:False for hand_type in hand_types}
            any_updated = False
            for hand_type in hand_types:
                squeeze_value = get_controller_squeeze_value(controller_bridge,hand_type)
                engaged = (
                    get_controller_matrix(controller_bridge,hand_type) is not None
                    and squeeze_value >= SG2_CONTROLLER_SQUEEZE_THRESHOLD
                )
                T_target = compute_sg2_controller_wrist_target(
                    bridge                    = controller_bridge,
                    hand_type                 = hand_type,
                    start_T_anchor            = start_T_anchor,
                    rotation_offset_R         = rotation_offset_R_by_hand[hand_type],
                    controller_back_offset_m  = SG2_CONTROLLER_BACK_OFFSET_M,
                    position_scale            = 1.0,) if engaged else None

                if T_target is None:
                    quest_missing_ticks[hand_type] += 1
                else:
                    T_quest_wrist[hand_type] = T_target
                    quest_missing_ticks[hand_type] = 0
                    quest_updated[hand_type] = True
                    any_updated = True
            quest_all_missing_ticks = 0 if any_updated else (quest_all_missing_ticks+1)

            # Attach controller wrist to SG2 wrist (same clutch/smoothing logic
            # as the hand-tracking notebook: blends in over attach_seconds once
            # the target is within attach_start_distance of the robot's wrist).
            arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
            (
                T_raw_wrist_target,
                T_attach_wrist_target,
                T_virtual_quest_wrist,
                attached_wrist_targets,
                quest_attach_state,
                last_solved_wrist_target,
                wrist_target_motion,
            ) = update_sg2_wrist_attach_targets(
                env                      = env,
                hand_types               = hand_types,
                T_quest_wrist            = T_quest_wrist,
                quest_missing_ticks      = quest_missing_ticks,
                quest_updated            = quest_updated,
                quest_attach_state       = quest_attach_state,
                last_solved_wrist_target = last_solved_wrist_target,
                max_missing_quest_ticks  = max_missing_quest_ticks,
                dt                       = dt,
                attach_seconds           = SG2_CONTROLLER_ATTACH_SECONDS,
                attach_start_distance    = SG2_CONTROLLER_ATTACH_START_DISTANCE,)

            # Trigger -> SG2 gripper (direct mapping, no pinch gesture needed).
            for hand_type in hand_types:
                engaged = quest_missing_ticks[hand_type] <= max_missing_quest_ticks
                if engaged:
                    trigger_value = get_controller_trigger_value(controller_bridge,hand_type)
                    gripper_q_by_hand[hand_type] = get_sg2_gripper_driver_qpos_from_trigger(trigger_value)
                set_sg2_gripper_qpos(env,hand_type,gripper_q_by_hand[hand_type])
                gripper_info_by_hand[hand_type] = {
                    "status": "tracking" if engaged else "disengaged",
                    "pinch_distance": np.nan,
                    "closed_ratio": float(np.clip(gripper_q_by_hand[hand_type] / 1.1,0.0,1.0)),
                    "q_driver": gripper_q_by_hand[hand_type],
                    "p_thumb_tip": None,
                    "p_index_tip": None,
                }

            # Collision-aware arm IK
            collision_ik_result = solve_sg2_collision_aware_wrist_ik(
                env                    = env,
                tracking_solver        = collision_ik_solvers["tracking"],
                depenetration_solver   = None,
                wrist_targets_by_hand  = attached_wrist_targets,
                collision_geom_pairs   = collision_geom_pairs,
                native_backend         = native_backend,
                safe_margin            = 0.05,
                signed_distance_topk   = 5,
                enable_depenetration   = False,)
            active_ik_solver = collision_ik_result["solver"]
            last_solved_wrist_target,arm_ik_info_by_hand = update_last_solved_wrist_targets(
                last_solved_wrist_target = last_solved_wrist_target,
                arm_ik_info_by_hand      = arm_ik_info_by_hand,
                attached_wrist_targets   = attached_wrist_targets,
                ik_info                  = collision_ik_result["ik_info"],)

            q_use = np.clip(env.get_qpos(joint_names=command_joint_names),q_use_min,q_use_max)
            env.forward(q=q_use,joint_names=command_joint_names)
            tmr_update.end()

        if tmr_render.do_run():
            draw_cmds = make_sg2_metaquest_draw_cmds(
                rp                    = rp,
                env                   = env,
                hand_types            = hand_types,
                hand_labels           = hand_labels,
                quest_missing_ticks   = quest_missing_ticks,
                T_quest_wrist         = T_quest_wrist,
                T_virtual_quest_wrist = T_virtual_quest_wrist,
                collision_ik_result   = collision_ik_result,
                active_ik_solver      = active_ik_solver,
                plot_signed_distance  = False,
                plot_ik_targets       = False,)
            overlay_cmds = [
                rp.overlay("Attach",make_sg2_attach_overlay_text(hand_types,hand_labels,quest_attach_state),loc="top left"),
                rp.overlay("Gripper",make_sg2_gripper_overlay_text(hand_types,hand_labels,gripper_info_by_hand),loc="top left"),
                rp.overlay("IK",make_sg2_ik_overlay_text(collision_ik_result),loc="top left"),
                rp.overlay("Collision",make_sg2_collision_overlay_text(collision_ik_result),loc="top left"),
            ]
            rp.render(env=env,overlays=overlay_cmds,draws=draw_cmds,mode="snapshot",profile=False)
            tmr_render.end()
            sleep(0.001)

except KeyboardInterrupt:
    print("Interrupted.")
finally:
    controller_bridge.stop()
    close_objects("rp")


[Render][569][35.3s] delayed! T:[50.0]ms interval:[55.6]ms prev_run:[1.5]ms.
Interrupted.


In [15]:
if getattr(env,"final_rgb_img",None) is not None:
    imshow(env.final_rgb_img,fig_w=6,title="Final scene",title_fs=8)
print("Done.")


Done.
